<a href="https://colab.research.google.com/github/nelcie3002/energy_demand_forecast/blob/main/energy_demand_forecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install lightgbm python-dotenv


In [3]:
import requests
import pandas as pd
from google.colab import userdata

API_KEY = userdata.get("EIA_API_KEY")

url = "https://api.eia.gov/v2/electricity/rto/region-data/data/"
params = {
    "api_key": API_KEY,
    "frequency": "hourly",
    "data[0]": "value",
    "facets[respondent][]": "CISO",
    "facets[type][]": "D",
    "start": "2025-01-01T00",
    "end": "2025-01-07T23",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "length": 5000,
}

r = requests.get(url, params=params)
print(r.status_code)
pd.DataFrame(r.json()["response"]["data"]).head()

200


,period,respondent,respondent-name,type,type-name,value,value-units
0,2025-01-01T00,CISO,California Independent System Operator,D,Demand,23877,megawatthours
1,2025-01-01T01,CISO,California Independent System Operator,D,Demand,22237,megawatthours
2,2025-01-01T02,CISO,California Independent System Operator,D,Demand,23890,megawatthours
3,2025-01-01T03,CISO,California Independent System Operator,D,Demand,26097,megawatthours
4,2025-01-01T04,CISO,California Independent System Operator,D,Demand,25742,megawatthours


In [4]:
import time

frames = []
offset = 0

while True:
    params = {
        "api_key": API_KEY,
        "frequency": "hourly",
        "data[0]": "value",
        "facets[respondent][]": "CISO",
        "facets[type][]": "D",
        "start": "2023-01-01T00",
        "end": "2025-12-31T23",
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
        "offset": offset,
        "length": 5000,
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()["response"]["data"]
    if not data:
        break
    frames.append(pd.DataFrame(data))
    offset += 5000
    time.sleep(0.5)  # politesse envers l'API

df = pd.concat(frames, ignore_index=True)
df["period"] = pd.to_datetime(df["period"])
df = df.set_index("period")[["value"]].astype(float)
df.columns = ["demand"]
df = df[~df.index.duplicated()]   # sécurité : supprime les doublons

print(df.shape)
print(df.index.min(), "->", df.index.max())
df.head()

(26304, 1)
2023-01-01 00:00:00 -> 2025-12-31 23:00:00


,demand
period,
2023-01-01 00:00:00,24007.0
2023-01-01 01:00:00,24546.0
2023-01-01 02:00:00,25344.0
2023-01-01 03:00:00,26468.0
2023-01-01 04:00:00,26047.0


In [5]:
full_index = pd.date_range(df.index.min(), df.index.max(), freq="h")
print("Heures manquantes :", len(full_index) - len(df))
df["demand"].describe()

Heures manquantes : 0


,demand
count,26302.000000
mean,25328.494335
std,4411.966172
min,11819.000000
25%,22215.000000
50%,24540.500000
75%,27201.000000
max,47571.000000


In [6]:
print(df[df["demand"].isna()])   # voir lesquelles

df["demand"] = df["demand"].interpolate(limit=3)   # interpolation linéaire
print("NaN restants :", df["demand"].isna().sum())

                     demand
period                     
2023-10-31 21:00:00     NaN
2023-11-14 20:00:00     NaN
NaN restants : 0


In [7]:
w = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
    "latitude": 34.05, "longitude": -118.24,   # Los Angeles
    "start_date": "2023-01-01", "end_date": "2025-12-31",
    "hourly": "temperature_2m,relative_humidity_2m",
    "timezone": "UTC",
}).json()["hourly"]

weather = pd.DataFrame(w)
weather["time"] = pd.to_datetime(weather["time"])
weather = weather.set_index("time")
weather.columns = ["temperature", "humidity"]

df = df.join(weather, how="left")
print(df.isna().sum())
df.head()

demand         0
temperature    0
humidity       0
dtype: int64


,demand,temperature,humidity
period,,,
2023-01-01 00:00:00,24007.0,14.3,93
2023-01-01 01:00:00,24546.0,14.1,94
2023-01-01 02:00:00,25344.0,13.9,94
2023-01-01 03:00:00,26468.0,13.9,95
2023-01-01 04:00:00,26047.0,13.8,97


In [8]:
import plotly.express as px

px.scatter(df.sample(3000, random_state=0), x="temperature", y="demand",
           opacity=0.4, title="Demande vs température (Los Angeles)").show()

In [9]:
!pip install holidays

In [10]:
import holidays

# 1. Passer en heure locale de Californie (pour les habitudes de consommation)
local = df.index.tz_localize("UTC").tz_convert("America/Los_Angeles")

df["hour"] = local.hour
df["dayofweek"] = local.dayofweek
df["month"] = local.month
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

# 2. Jours fériés californiens
us_holidays = holidays.US(state="CA", years=[2023, 2024, 2025])
df["is_holiday"] = pd.Series(local.date, index=df.index).isin(us_holidays).astype(int)

# 3. Décalages de la demande (jamais de valeurs futures)
df["lag_24"] = df["demand"].shift(24)
df["lag_168"] = df["demand"].shift(168)
df["rolling_24"] = df["demand"].shift(24).rolling(24).mean()

# 4. Température : décalage et seuil de climatisation
df["temp_lag_24"] = df["temperature"].shift(24)
df["cooling_degrees"] = (df["temperature"] - 18).clip(lower=0)

df = df.dropna()
print(df.shape)
df.head()

(26136, 13)


,demand,temperature,humidity,hour,dayofweek,month,is_weekend,is_holiday,lag_24,lag_168,rolling_24,temp_lag_24,cooling_degrees
period,,,,,,,,,,,,,
2023-01-08 00:00:00,22131.0,16.9,66,16,5,1,1,0,23297.0,24007.0,24271.041667,14.7,0.0
2023-01-08 01:00:00,22930.0,13.6,80,17,5,1,1,0,23581.0,24546.0,24201.375000,11.9,0.0
2023-01-08 02:00:00,24079.0,13.5,78,18,5,1,1,0,25216.0,25344.0,24171.583333,11.5,0.0
2023-01-08 03:00:00,25792.0,13.7,73,19,5,1,1,0,27226.0,26468.0,24155.416667,10.3,0.0
2023-01-08 04:00:00,25704.0,11.6,83,20,5,1,1,0,27141.0,26047.0,24126.541667,10.1,0.0


In [11]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# Séparation chronologique : entraînement jusqu'à fin juin 2025, test ensuite
train = df[:"2025-06-30"]
test = df["2025-07-01":]

X_cols = [c for c in df.columns if c != "demand"]
print("Variables :", X_cols)
print("Train :", train.shape, "| Test :", test.shape)

# Baseline naïve : la demande de la même heure la semaine dernière
baseline = test["lag_168"]
print(f"Baseline  -> MAE : {mean_absolute_error(test['demand'], baseline):.0f} MWh | "
      f"MAPE : {mean_absolute_percentage_error(test['demand'], baseline):.2%}")

# Modèle LightGBM
model = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.03,
                          num_leaves=31, random_state=42, verbose=-1)
model.fit(
    train[X_cols], train["demand"],
    eval_set=[(test[X_cols], test["demand"])],
    eval_metric="mae",
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
)

pred = model.predict(test[X_cols])
print(f"LightGBM  -> MAE : {mean_absolute_error(test['demand'], pred):.0f} MWh | "
      f"MAPE : {mean_absolute_percentage_error(test['demand'], pred):.2%}")

Variables : ['temperature', 'humidity', 'hour', 'dayofweek', 'month', 'is_weekend', 'is_holiday', 'lag_24', 'lag_168', 'rolling_24', 'temp_lag_24', 'cooling_degrees']
Train : (21720, 13) | Test : (4416, 13)
Baseline  -> MAE : 1766 MWh | MAPE : 6.22%
Training until validation scores don't improve for 100 rounds
[200]	valid_0's l1: 752.645	valid_0's l2: 1.13102e+06
Early stopping, best iteration is:
[231]	valid_0's l1: 746.969	valid_0's l2: 1.11936e+06
LightGBM  -> MAE : 747 MWh | MAPE : 2.71%


In [12]:
import plotly.graph_objects as go
import plotly.express as px

res = test[["demand"]].copy()
res["pred"] = pred
res["error"] = res["demand"] - res["pred"]
res["abs_pct_error"] = (res["error"].abs() / res["demand"]) * 100

# 1. Réel vs prévu sur deux semaines
zoom = res["2025-07-14":"2025-07-27"]
fig = go.Figure()
fig.add_scatter(x=zoom.index, y=zoom["demand"], name="Réel")
fig.add_scatter(x=zoom.index, y=zoom["pred"], name="Prévu")
fig.update_layout(title="Demande électrique CISO : réel vs prévu (2 semaines, juillet 2025)",
                  yaxis_title="MWh")
fig.show()

# 2. Importance des variables
imp = pd.Series(model.feature_importances_, index=X_cols).sort_values()
px.bar(imp, orientation="h", title="Importance des variables").show()

In [13]:
# Erreur par heure locale
local_test = res.index.tz_localize("UTC").tz_convert("America/Los_Angeles")
res["hour_local"] = local_test.hour
px.bar(res.groupby("hour_local")["abs_pct_error"].mean(),
       title="Erreur moyenne (%) par heure locale").show()

# Erreur par mois
print(res.groupby(res.index.month)["abs_pct_error"].mean().round(2))

# Les 10 pires heures
print(res.nlargest(10, "abs_pct_error")[["demand", "pred", "abs_pct_error"]])

period
7     2.83
8     2.72
9     2.91
10    2.45
11    2.93
12    2.45
Name: abs_pct_error, dtype: float64
                      demand          pred  abs_pct_error
period                                                   
2025-07-31 21:00:00  11819.0  31240.941870     164.328132
2025-08-01 21:00:00  29807.0  20804.197543      30.203652
2025-11-21 21:00:00  23273.0  28413.618008      22.088334
2025-11-21 22:00:00  23366.0  27807.034253      19.006395
2025-11-21 19:00:00  23730.0  28238.421557      18.998827
2025-10-04 20:00:00  22784.0  26802.678070      17.638159
2025-10-04 23:00:00  21708.0  25510.853277      17.518211
2025-11-21 20:00:00  23954.0  28123.587007      17.406642
2025-10-04 21:00:00  22557.0  26425.086872      17.148055
2025-10-04 22:00:00  22482.0  26002.011740      15.657022


In [14]:
print(df["2025-07-31 18:00":"2025-08-01 02:00"][["demand", "lag_24"]])

                      demand   lag_24
period                               
2025-07-31 18:00:00  30767.0  31144.0
2025-07-31 19:00:00  30296.0  31537.0
2025-07-31 20:00:00  29881.0  31227.0
2025-07-31 21:00:00  11819.0  30563.0
2025-07-31 22:00:00  29886.0  29651.0
2025-07-31 23:00:00  30724.0  30255.0
2025-08-01 00:00:00  31862.0  31194.0
2025-08-01 01:00:00  33033.0  32347.0
2025-08-01 02:00:00  34012.0  33668.0


In [15]:
raw = df[["demand", "temperature", "humidity"]].copy()

# Correction ciblée de l'anomalie identifiée
raw.loc["2025-07-31 21:00", "demand"] = None
raw["demand"] = raw["demand"].interpolate(limit=3)
print(raw.loc["2025-07-31 20:00":"2025-07-31 22:00", "demand"])

period
2025-07-31 20:00:00    29881.0
2025-07-31 21:00:00    29883.5
2025-07-31 22:00:00    29886.0
Name: demand, dtype: float64


In [16]:
med = raw["demand"].rolling(25, center=True, min_periods=5).median()
outliers = (raw["demand"] - med).abs() / med > 0.35
print("Valeurs suspectes :", outliers.sum())
raw[outliers]

Valeurs suspectes : 32


,demand,temperature,humidity
period,,,
2023-07-02 03:00:00,35936.0,19.5,75
2023-07-02 04:00:00,35914.0,18.2,81
2023-07-03 03:00:00,35393.0,19.9,73
2023-07-16 03:00:00,38873.0,22.6,62
2023-07-16 04:00:00,38837.0,21.4,68
2023-07-31 03:00:00,37878.0,24.7,72
2023-08-06 03:00:00,37457.0,22.9,67
2023-08-06 04:00:00,36854.0,21.7,72
2023-08-07 02:00:00,37642.0,25.5,59


In [17]:
def build_features(d):
    d = d.copy()
    local = d.index.tz_localize("UTC").tz_convert("America/Los_Angeles")
    d["hour"] = local.hour
    d["dayofweek"] = local.dayofweek
    d["month"] = local.month
    d["is_weekend"] = (d["dayofweek"] >= 5).astype(int)
    d["is_holiday"] = pd.Series(local.date, index=d.index).isin(us_holidays).astype(int)
    d["lag_24"] = d["demand"].shift(24)
    d["lag_168"] = d["demand"].shift(168)
    d["rolling_24"] = d["demand"].shift(24).rolling(24).mean()
    d["temp_lag_24"] = d["temperature"].shift(24)
    d["cooling_degrees"] = (d["temperature"] - 18).clip(lower=0)
    return d.dropna()

df2 = build_features(raw)
print(df2.shape)

(25968, 13)


In [18]:
train = df2[:"2025-03-31"]
valid = df2["2025-04-01":"2025-08-31"]
test  = df2["2025-09-01":]

X_cols = [c for c in df2.columns if c != "demand"]

baseline = test["lag_168"]
print(f"Baseline -> MAE : {mean_absolute_error(test['demand'], baseline):.0f} | "
      f"MAPE : {mean_absolute_percentage_error(test['demand'], baseline):.2%}")

model = lgb.LGBMRegressor(n_estimators=3000, learning_rate=0.03,
                          num_leaves=31, random_state=42, verbose=-1)
model.fit(train[X_cols], train["demand"],
          eval_set=[(valid[X_cols], valid["demand"])],
          eval_metric="mae",
          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])

pred = model.predict(test[X_cols])
print(f"LightGBM -> MAE : {mean_absolute_error(test['demand'], pred):.0f} | "
      f"MAPE : {mean_absolute_percentage_error(test['demand'], pred):.2%}")

Baseline -> MAE : 1475 | MAPE : 5.50%
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[479]	valid_0's l1: 807.355	valid_0's l2: 1.2006e+06
LightGBM -> MAE : 699 | MAPE : 2.64%


In [19]:
imp = pd.Series(model.booster_.feature_importance(importance_type="gain"),
                index=X_cols).sort_values()
px.bar(imp, orientation="h", title="Importance des variables (gain)").show()

In [20]:
med = raw["demand"].rolling(25, center=True, min_periods=5).median()
outliers = (raw["demand"] - med).abs() / med > 0.5
print("Valeurs suspectes :", outliers.sum())
raw[outliers]

Valeurs suspectes : 0


,demand,temperature,humidity
period,,,


In [21]:
res = test[["demand"]].copy()
res["pred"] = pred
res["error"] = res["demand"] - res["pred"]
res["abs_pct_error"] = res["error"].abs() / res["demand"] * 100

# Erreur par heure locale
local_test = res.index.tz_localize("UTC").tz_convert("America/Los_Angeles")
res["hour_local"] = local_test.hour
by_hour = res.groupby("hour_local")["abs_pct_error"].mean()
print("Pic d'erreur : heure locale", by_hour.idxmax(), "->", round(by_hour.max(), 2), "%")
print("Erreur minimale : heure locale", by_hour.idxmin(), "->", round(by_hour.min(), 2), "%")
px.bar(by_hour, title="Erreur moyenne (%) par heure locale").show()

# Erreur par mois
print(res.groupby(res.index.month)["abs_pct_error"].mean().round(2))

# Pires heures (avec l'heure locale et le jour de la semaine)
worst = res.nlargest(10, "abs_pct_error").copy()
idx_local = worst.index.tz_localize("UTC").tz_convert("America/Los_Angeles")
worst["local"] = idx_local.strftime("%Y-%m-%d %H:%M")
worst["jour"] = idx_local.day_name()
worst[["local", "jour", "demand", "pred", "abs_pct_error"]].round(1)

Pic d'erreur : heure locale 14 -> 4.11 %
Erreur minimale : heure locale 2 -> 1.73 %


period
9     2.80
10    2.47
11    2.94
12    2.35
Name: abs_pct_error, dtype: float64


,local,jour,demand,pred,abs_pct_error
period,,,,,
2025-11-21 21:00:00,2025-11-21 13:00,Friday,23273.0,28235.0,21.3
2025-11-21 19:00:00,2025-11-21 11:00,Friday,23730.0,28256.6,19.1
2025-11-21 22:00:00,2025-11-21 14:00,Friday,23366.0,27607.7,18.2
2025-11-21 20:00:00,2025-11-21 12:00,Friday,23954.0,28049.0,17.1
2025-10-04 20:00:00,2025-10-04 13:00,Saturday,22784.0,26358.9,15.7
2025-11-11 21:00:00,2025-11-11 13:00,Tuesday,28065.0,23690.6,15.6
2025-10-04 21:00:00,2025-10-04 14:00,Saturday,22557.0,25848.4,14.6
2025-09-23 01:00:00,2025-09-22 18:00,Monday,36141.0,30944.3,14.4
2025-11-14 19:00:00,2025-11-14 11:00,Friday,25601.0,29253.1,14.3


In [22]:
zoom = res["2025-10-06":"2025-10-19"]
fig = go.Figure()
fig.add_scatter(x=zoom.index, y=zoom["demand"], name="Réel")
fig.add_scatter(x=zoom.index, y=zoom["pred"], name="Prévu")
fig.update_layout(title="Demande électrique CISO : réel vs prévu (2 semaines, octobre 2025)",
                  yaxis_title="MWh")
fig.show()